# 02 - SA-CVA Sensitivities

Synthetic engineering and validation evidence, not final regulatory capital.

Use this notebook with the [CVA package journey](../docs/PACKAGE_JOURNEY.md) and the executable [package demo](../examples/run_demo.py).

Raw inputs your upstream risk engine must emit: counterparty rows, netting-set rows, eligible hedge rows where applicable, and SA-CVA sensitivity rows for approved desks. The package dataset contract defines the fixture files and Arrow column-spec handoff: [`DATASET_CONTRACT.md`](../docs/DATASET_CONTRACT.md). The staged usage path is described in the [CVA package journey](../docs/PACKAGE_JOURNEY.md).

This notebook runs the synthetic SA-CVA sensitivity set across supported delta and vega risk classes and shows fail-closed behavior for an unsupported CCS vega input.


## Flow

```mermaid
flowchart LR
    A[SA-CVA sensitivity rows] --> B[Delta and vega risk classes]
    B --> C[Public CVA row API]
    C --> D[Risk-class aggregation records]
    D --> E[Capital result evidence]
    B --> F[Unsupported CCS vega input]
    F --> G[Fail-closed validation]
```


In [ ]:
from pathlib import Path
import sys

HERE = Path.cwd()
PACKAGE_ROOT = None
REPO_ROOT = None

for candidate in (HERE, *HERE.parents):
    if (candidate / "src" / "frtb_cva").exists():
        PACKAGE_ROOT = candidate
        REPO_ROOT = candidate.parent.parent if candidate.parent.name == "packages" else candidate
        break
    package_candidate = candidate / "packages" / "frtb-cva"
    if (package_candidate / "src" / "frtb_cva").exists():
        REPO_ROOT = candidate
        PACKAGE_ROOT = package_candidate
        break

if PACKAGE_ROOT is None or REPO_ROOT is None:
    raise RuntimeError("Could not locate the frtb-cva package root")

workspace_src_paths = tuple(sorted((REPO_ROOT / "packages").glob("*/src")))
for path in (
    PACKAGE_ROOT / "examples",
    PACKAGE_ROOT,
    PACKAGE_ROOT / "src",
    *workspace_src_paths,
    REPO_ROOT,
):
    text = str(path)
    if text not in sys.path:
        sys.path.insert(0, text)

try:
    from IPython.display import Markdown, display
except ImportError:

    class Markdown(str):
        pass

    def display(value: object) -> None:
        print(value)


PACKAGE_ROOT

## Public API happy path

This notebook runs synthetic SA-CVA sensitivities through the top-level row API: `calculate_cva_capital`.


In [ ]:
from dataclasses import replace

from cva_notebook_data import markdown_table, notebook_context, sample_sa_sensitivities
from frtb_cva import (
    CvaInputError,
    CvaMethod,
    SaCvaRiskClass,
    SaCvaRiskMeasure,
    calculate_cva_capital,
    validate_cva_result_reconciliation,
)

sensitivities = sample_sa_sensitivities()
context = notebook_context(method=CvaMethod.SA_CVA, run_id="cva-sa-notebook", sa_cva_approved=True)
result = calculate_cva_capital(context, (), (), sensitivities=sensitivities)
validate_cva_result_reconciliation(result)

display(
    Markdown(
        markdown_table(
            ("metric", "value"),
            (
                ("total CVA capital", f"{result.total_cva_capital:,.2f}"),
                ("risk-class result records", len(result.sa_cva_risk_class_capitals)),
                ("input hash", result.input_hash[:16]),
            ),
        )
    )
)

## Implementation details / math verification - SA-CVA risk classes and unsupported features

The remaining cells inspect risk-class aggregation records and fail-closed behavior for unsupported inputs.


In [ ]:
risk_class_rows = tuple(
    (
        item.risk_class.value,
        item.risk_measure.value,
        len(item.bucket_capitals),
        f"{item.pre_multiplier_capital:,.2f}",
        f"{item.post_multiplier_capital:,.2f}",
        ", ".join(item.citations[:3]),
    )
    for item in result.sa_cva_risk_class_capitals
)

display(
    Markdown(
        markdown_table(
            ("risk class", "measure", "buckets", "pre multiplier", "post multiplier", "citations"),
            risk_class_rows,
        )
    )
)

In [ ]:
unsupported_ccs_vega = replace(
    next(
        item
        for item in sensitivities
        if item.risk_class is SaCvaRiskClass.COUNTERPARTY_CREDIT_SPREAD
    ),
    sensitivity_id="sens-ccs-vega-unsupported",
    risk_measure=SaCvaRiskMeasure.VEGA,
    volatility_input=0.2,
)

try:
    calculate_cva_capital(context, (), (), sensitivities=(unsupported_ccs_vega,))
except CvaInputError as exc:
    unsupported_message = str(exc)

display(
    Markdown(
        markdown_table(
            ("unsupported input", "fail-closed message"),
            (("CCS vega", unsupported_message),),
        )
    )
)

## See also

- [CVA package journey](../docs/PACKAGE_JOURNEY.md)
- [CVA dataset contract](../docs/DATASET_CONTRACT.md)
- [Client integration guide](../../../docs/CLIENT_INTEGRATION.md)
- [SBM package journey](../../frtb-sbm/docs/PACKAGE_JOURNEY.md)
- [DRC package journey](../../frtb-drc/docs/PACKAGE_JOURNEY.md)
- [RRAO package journey](../../frtb-rrao/docs/PACKAGE_JOURNEY.md)
- [IMA package journey](../../frtb-ima/docs/PACKAGE_JOURNEY.md)
